# CP3-07 检查点回放 Replay：从历史位置重新执行

本节使用 `PostgresSaver` 演示 replay。前两个代码单元在同一个 `thread_id` 上各启动一次正常运行；第三个代码单元从历史中找到“`node_poem` 与 `node_joke` 即将执行”的快照，并使用它的 `config` 重新向后推进。

## Replay 的核心调用

`graph.invoke(None, config=replay_checkpoint.config)` 中的 `None` 表示不提交新的输入，`checkpoint_id` 则决定从哪一个历史快照开始。代码用集合比较 `next`，避免并行任务的调度顺序在不同运行中发生变化。

运行前需要 PostgreSQL、`langgraph-checkpoint-postgres`、`psycopg` 与 DeepSeek API。与 CP3-04～06 不同，本节不制造错误，而是观察从历史分支重新执行两个 LLM 节点的效果。


In [ ]:
import os
from typing import TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, StateGraph
from loguru import logger

load_dotenv(override=True)
MODEL_NAME = os.getenv('DEEPSEEK_MODEL', 'deepseek-v4-flash')
DB_URL = os.getenv('LANGGRAPH_DB_URL')
if not os.getenv('DEEPSEEK_API_KEY'):
    raise RuntimeError('缺少 DEEPSEEK_API_KEY，请先配置 .env 或系统环境变量。')
if not DB_URL:
    raise RuntimeError('缺少 LANGGRAPH_DB_URL，请配置 PostgreSQL 连接串。')
model = ChatDeepSeek(model=MODEL_NAME, extra_body={'thinking': {'type': 'disabled'}})

class OverAllState(TypedDict, total=False):
    topic: str
    poem: str
    joke: str
    final_output: str

class InputState(TypedDict):
    topic: str

class OutputState(TypedDict):
    final_output: str

topics = ['布偶猫', '狸花猫', '金渐层']
topic_index = 0

def node_change_topic(state: InputState) -> OverAllState:
    global topic_index
    logger.info('topic_index: {}', topic_index)
    sub_topic = topics[topic_index]
    topic_index = (topic_index + 1) % len(topics)
    return {'topic': f'{state["topic"]}:{sub_topic}'}

def node_poem(state: OverAllState) -> OverAllState:
    logger.info('node_poem 正在执行')
    response = model.invoke([HumanMessage(content=f'写一首关于{state["topic"]}主题的七言绝句')])
    return {'poem': response.content}

def node_joke(state: OverAllState) -> OverAllState:
    logger.info('node_joke 正在执行')
    response = model.invoke([HumanMessage(content=f'写一个关于{state["topic"]}主题的笑话')])
    return {'joke': response.content}

def node_output(state: OverAllState) -> OutputState:
    logger.info('node_output 正在执行')
    return {
        'final_output': (
            f'关于{state["topic"]}的七言绝句:{state["poem"]}'
            + chr(10)
            + f'笑话:{state["joke"]}'
        )
    }

# node_change_topic 完成后，诗和笑话进入同一个并行超步，最后再汇合到 node_output。
builder = StateGraph(state_schema=OverAllState, input_schema=InputState, output_schema=OutputState)
builder.add_node('node_change_topic', node_change_topic)
builder.add_node('node_poem', node_poem)
builder.add_node('node_joke', node_joke)
builder.add_node('node_output', node_output)
builder.add_edge(START, 'node_change_topic')
builder.add_edge('node_change_topic', 'node_poem')
builder.add_edge('node_change_topic', 'node_joke')
builder.add_edge('node_poem', 'node_output')
builder.add_edge('node_joke', 'node_output')
builder.add_edge('node_output', END)

from langgraph.checkpoint.postgres import PostgresSaver
THREAD_ID = os.getenv('CP3_REPLAY_THREAD_ID', 'chapter03-08')
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)
    print(graph.get_graph().draw_mermaid())
    config = {'configurable': {'thread_id': THREAD_ID}}
    first_result = graph.invoke({'topic': '猫咪'}, config=config)
    print(first_result)


In [ ]:
# 使用同一个 thread_id 再启动一次运行，给历史增加另一条完整执行链。
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)
    config = {'configurable': {'thread_id': THREAD_ID}}
    second_result = graph.invoke({'topic': '猫咪'}, config=config)
    print(second_result)


## 为什么要先找 `next == ('node_poem', 'node_joke')`

检查点的 `next` 表示下一个超步待执行的节点。由于 `node_change_topic` 向两个节点扇出，诗和笑话会同时出现在这个元组中；它正是“主题已确定、两个生成任务尚未执行”的回放起点。

Replay 不等于缓存命中：这次起点在两个 LLM 节点之前，因此诗和笑话会重新调用模型，结果可能与第一次运行不同。检查点保证的是执行拓扑和状态起点，而不是随机模型文本完全一致。


In [ ]:
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)
    config = {'configurable': {'thread_id': THREAD_ID}}
    history_checkpoints = list(graph.get_state_history(config=config))

    target_next = {'node_poem', 'node_joke'}
    replay_checkpoint = next(
        (snapshot for snapshot in history_checkpoints if set(snapshot.next) == target_next),
        None,
    )
    if replay_checkpoint is None:
        raise RuntimeError('没有找到 node_poem/node_joke 的并行检查点，请先执行前两个代码单元。')

    print({
        'checkpoint_id': replay_checkpoint.config['configurable'].get('checkpoint_id'),
        'next': replay_checkpoint.next,
        'topic': replay_checkpoint.values.get('topic'),
    })

    # input=None + 历史 config：从选中的检查点重放剩余节点。
    replay_result = graph.invoke(None, config=replay_checkpoint.config)
    print(replay_result)


## 与 CP3-06 的区别

- CP3-06 是“失败后恢复”：从当前线程最新的失败检查点继续执行未完成任务。
- CP3-07 是“主动历史回放”：挑选某个旧 `checkpoint_id`，观察从那个时间点再次向后运行。

二者都依赖 `StateSnapshot` 的 `values`、`next`、`config`、`metadata`、`tasks` 与 `parent_config`。生产系统若要求幂等，还需要给模型调用增加缓存或请求去重。
